In [33]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import f_oneway

# Đọc dữ liệu
df = pd.read_csv("cleaned_orders_with_2_items.csv")


In [5]:

# Xử lý dữ liệu
# 1. Chuyển đổi Order Placed At thành các yếu tố thời gian
df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y')
df['Hour'] = df['Order Placed At'].dt.hour
df['DayOfWeek'] = df['Order Placed At'].dt.dayofweek
df['Month'] = df['Order Placed At'].dt.month

# 2. Vector hóa 'Items in order'
tfidf = TfidfVectorizer()
X_items = tfidf.fit_transform(df['Items in order'])

# 3. One-hot Encoding cho 'Restaurant ID'
encoder = OneHotEncoder(sparse_output=False)
X_restaurant = encoder.fit_transform(df[['Restaurant name']])

# 4. Các đặc trưng khác
X = pd.concat([pd.DataFrame(X_items.toarray()), pd.DataFrame(X_restaurant)], axis=1)
df['Hour'] = df['Order Placed At'].dt.hour
df['DayOfWeek'] = df['Order Placed At'].dt.dayofweek
df['Month'] = df['Order Placed At'].dt.month


In [41]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Biến mục tiêu
y = df['KPT duration (minutes)']

# Chia dữ liệu thành train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Mô hình RandomForest
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Dự đoán và đánh giá
y_pred = model.predict(X_test)

# Tính MSE
mse = mean_squared_error(y_test, y_pred)



# Tính RMSE (căn bậc 2 của MSE)
rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.4f}")

# Tính R2 (R-squared)
r2 = r2_score(y_test, y_pred)
print(f"R-squared (R²): {r2:.4f}")




RMSE: 4.3172
R-squared (R²): 0.0410


In [42]:
print(X.columns)

Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       145, 146, 147, 148,   0,   1,   2,   3,   4,   5],
      dtype='int64', length=155)


In [43]:
sample_orders = [
    {"Restaurant name": "Swaad", "Items in order": "1 x Grilled Chicken Jamaican Tender, 1 x Grilled Chicken Peri Peri Tangdi", "Order Placed At": "11:38 PM, September 10 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Fried Chicken Ghostbuster Tender, 1 x Angara Grilled Paneer (8 pcs)", "Order Placed At": "03:45 PM, September 10 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Peri Peri Krispers, 1 x Fried Chicken Angara Tender", "Order Placed At": "03:04 PM, September 10 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Grilled Chicken Jamaican Tangdi, 1 x Bone in Jamaican Grilled Chicken", "Order Placed At": "12:28 PM, September 10 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Fried Chicken Angara Tender, 1 x Angara Rice", "Order Placed At": "10:51 PM, September 09 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Animal Fries, 1 x Salted Fries", "Order Placed At": "06:09 PM, September 07 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Bone in Jamaican Grilled Chicken, 1 x Fried Chicken Angara Tender", "Order Placed At": "04:00 PM, September 07 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Bone in Angara Grilled Chicken, 1 x Bone in Jamaican Grilled Chicken", "Order Placed At": "09:50 PM, September 06 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Bone in Smoky Bbq Grilled Chicken, 1 x Fried Chicken Peri Peri Tenders + Peri Peri Fries", "Order Placed At": "02:03 AM, September 06 2024"},
    {"Restaurant name": "Swaad", "Items in order": "1 x Angara Aloo Tuk Tuki, 1 x AAC Signature Fries", "Order Placed At": "01:54 AM, September 06 2024"}
]


In [29]:
# Chuyển đổi sample_orders thành dataframe
sample_df = pd.DataFrame(sample_orders)
sample_df['Order Placed At'] = pd.to_datetime(sample_df['Order Placed At'])
sample_df['Hour'] = sample_df['Order Placed At'].dt.hour
sample_df['DayOfWeek'] = sample_df['Order Placed At'].dt.dayofweek
sample_df['Month'] = sample_df['Order Placed At'].dt.month

# Vector hóa 'Items in order'
X_sample_items = tfidf.transform(sample_df['Items in order'])

# One-hot Encoding cho 'Restaurant name'
X_sample_restaurant = encoder.transform(sample_df[['Restaurant name']])

# Các đặc trưng khác
X_sample = pd.concat([pd.DataFrame(X_sample_items.toarray()), pd.DataFrame(X_sample_restaurant)], axis=1)


/var/folders/93/jzhc9xv925g12nywcpx4b7b80000gn/T/ipykernel_14636/1389608380.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sample_df['Order Placed At'] = pd.to_datetime(sample_df['Order Placed At'])


In [30]:
# Dự đoán KPT duration cho các đơn hàng mẫu
X_sample.columns = X_sample.columns.astype(str)

y_sample_pred = model.predict(X_sample)

# In kết quả dự đoán
for i, pred in enumerate(y_sample_pred):
    print(f"Order {i+1}: Dự đoán KPT duration = {pred:.2f} minutes")

Order 1: Dự đoán KPT duration = 19.86 minutes
Order 2: Dự đoán KPT duration = 20.57 minutes
Order 3: Dự đoán KPT duration = 16.23 minutes
Order 4: Dự đoán KPT duration = 19.47 minutes
Order 5: Dự đoán KPT duration = 15.87 minutes
Order 6: Dự đoán KPT duration = 16.01 minutes
Order 7: Dự đoán KPT duration = 18.28 minutes
Order 8: Dự đoán KPT duration = 18.63 minutes
Order 9: Dự đoán KPT duration = 17.84 minutes
Order 10: Dự đoán KPT duration = 23.25 minutes


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [25]:
# Kiểm tra các cột trong X_train và X_sample
print(X_train.columns)
print(X_sample.columns)


Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '145', '146', '147', '148', '0', '1', '2', '3', '4', '5'],
      dtype='object', length=155)
Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '145', '146', '147', '148', '0', '1', '2', '3', '4', '5'],
      dtype='object', length=155)
